In [1]:
import sys
print(sys.version)

3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]


In [2]:
print("ok")

ok


In [3]:
%pwd

'd:\\companyproject\\aiassistant\\Medical_Assitant\\research'

In [4]:
import os
os.chdir("../")

In [5]:
%pwd

'd:\\companyproject\\aiassistant\\Medical_Assitant'

In [6]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\ASUS\.conda\envs\medibot2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Extract text from PDF files in the "data" directory

def load_pdf_file(data):
    loader = DirectoryLoader(data,
                             glob="*.pdf",
                             loader_cls=PyPDFLoader
                             )
    
    documents = loader.load()
    
    return documents
    



In [9]:
extracted_data = load_pdf_file(data ='Data/')

In [10]:
# extracted_data

In [11]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    
    text_chunks = text_splitter.split_documents(extracted_data)
    
    return text_chunks

In [12]:
text_chunks = text_split(extracted_data)
print("Number of text chunks:", len(text_chunks))


Number of text chunks: 2345


In [13]:
# text_chunks


In [12]:
# from langchain_huggingface import HuggingFaceEmbeddings

# %pip install sentence-transformers
# import sys
# !{sys.executable} -m pip install -U \
#     sentence-transformers==2.2.2 \
#     huggingface-hub==0.16.4

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [15]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "300"
embeddings = download_hugging_face_embeddings()

In [16]:
query_results = embeddings.embed_query("Hello world")
print("Query results:", len(query_results)) 

Query results: 384


In [17]:
# query_results


In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")



In [20]:
from pinecone import Pinecone, ServerlessSpec  # no .grpc import needed

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "hospital-assistant"
pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': 'cfe27ad6f7e28d314f37385afdd09b7d', 'date': 'Wed, 24 Jun 2026 11:53:27 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [21]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY



In [22]:
index_name = "hospital-assistant"

In [40]:
# EMBEDEB EACH CHUNK AND STORE IT IN PINECONE
from langchain_pinecone import PineconeVectorStore

docSearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [23]:
# LOAD EXISTING INDEX

from langchain_pinecone import PineconeVectorStore

docSearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)


In [24]:
retriever = docSearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [25]:
retrieved_docs = retriever.invoke("what is Enteric Fever?")

In [26]:
retrieved_docs

[Document(id='af1920e5-e326-4981-9b54-74afabacf3b4', metadata={'author': 'Tierney, Lawrence M.; Saint, Sanjay.; Whooley, Mary A.', 'creationdate': '2010-12-22T13:15:56+00:00', 'creator': 'PyPDF', 'keywords': '0071637907\r\n9780071637909', 'moddate': '2018-08-02T09:28:12+05:30', 'page': 225.0, 'page_label': '213', 'producer': 'PyPDF', 'source': 'Data\\Medical_book.pdf', 'subject': 'McGraw-Hill Medical', 'title': 'Current Essentials of Medicine', 'total_pages': 607.0}, page_content='Chapter 8 Infectious Diseases 213\n8\nEnteric Fever (Typhoid Fever)\n■ Essentials of Diagnosis\n• Caused by several Salmonellaspecies; in “typhoid fever,” serotype\nSalmonella typhi is causative and accompanied by bacteremia\n• Transmitted by contaminated food or drink; incubation period is\n5–21 days\n• Gradual onset of malaise, headache, and abdominal pain, fol-\nlowed by diarrhea, or with S. typhi, constipation; stepladder rise\nof fever to a maximum of 40°C over 7–10 days, then slow return'),
 Document(id

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.4,
    max_tokens=500,
    model="nvidia/nemotron-3-ultra-550b-a55b:free",  # ← the working free model
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

In [28]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering task."
    "Use the following piece of retirival context to answer the question at the end."
    "If you don't know the answer, just say you don't know. Don't try to make up an answer."
    "Use three sentences maximum. and  keep the answer as concise as possible."
    "\n\n"
    "{context}"
    
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human","{input}"),
    ]
)



In [29]:
question_answer_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is Enteric Fever?"})
print(response["answer"]) 

In [36]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(
    
    openai_api_key=os.environ.get("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    
)

# Rest of your code stays exactly the same
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "what is math?"})
print(response["answer"])

I don't know.


In [16]:
# %pip install pinecone

# import sys
# !{sys.executable} -m pip install -U langchain langchain-pinecone pinecone

# import sys
# !{sys.executable} -m pip install -U langchain-community

# %pip install protobuf


# %pip install googleapis-common-protos

# %pip install grpcio grpcio-tools googleapis-common-protos

# %pip install "pinecone[grpc]"


# %pip install langchain langchain-community

# %pip install --upgrade langchain langchain-core langchain-community

# import langchain
# print(langchain.__version__)

# Step 1: Check current versions
# import subprocess
# result = subprocess.run(['pip', 'show', 'langchain'], capture_output=True, text=True)
# print(result.stdout)

# Step 2: Reinstall in the correct conda environment
# import sys
# !{sys.executable} -m pip install langchain langchain-community langchain-core

# import sys
# !{sys.executable} -m pip install "langchain==0.3.25" "langchain-community==0.3.24" "langchain-core==0.3.59" --upgrade

import sys
!{sys.executable} -m pip install langchain-huggingface==0.1.2


  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
Using cached sentence_transformers-5.6.0-py3-none-any.whl (596 kB)

  Attempting uninstall: sentence-transformers

    Found existing installation: sentence-transformers 2.2.2

    Uninstalling sentence-transformers-2.2.2:

      Successfully uninstalled sentence-transformers-2.2.2

   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-trans